# Experiment 9 – Sentiment Analysis Using RNN

### Deep Learning Laboratory – TensorFlow/Keras

**Aim:** To implement sentiment analysis using a Recurrent Neural Network (RNN) with TensorFlow/Keras and classify text reviews as positive or negative.

### Learning Objectives
- Understand the basic idea of sentiment analysis.
- Understand how text is converted into numerical sequences.
- Learn the role of Embedding and SimpleRNN layers.
- Build and train an RNN for binary text classification.
- Evaluate the trained model and test it on new sentences.

## 1. Theory

**Sentiment Analysis** is the process of identifying the emotional opinion expressed in text. In this experiment, a review is classified as either **Positive** or **Negative**.

An **RNN (Recurrent Neural Network)** is useful for sequential data such as sentences because it processes words in sequence and maintains information from previous words.

### Basic Flow

```text
Text Review
    ↓
Tokenization
    ↓
Integer Sequences
    ↓
Padding
    ↓
Embedding
    ↓
SimpleRNN
    ↓
Dense + Sigmoid
    ↓
Positive / Negative
```

### Important Layers

**Embedding:** Converts word numbers into dense numerical vectors.

**SimpleRNN:** Reads the sequence and learns patterns from the order of words.

**Dense + Sigmoid:** Produces a probability between 0 and 1 for binary classification.

In [ ]:
# Step 1: Import required libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## 2. Load the IMDB Movie Review Dataset

Keras provides the **IMDB Movie Reviews** dataset for sentiment classification.

The labels are:

- `0` → Negative review
- `1` → Positive review

To keep the laboratory experiment manageable, we use a subset of the dataset.

In [ ]:
# Step 2: Load IMDB dataset
VOCAB_SIZE = 10000
MAX_LEN = 200

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(
    num_words=VOCAB_SIZE
)

# Use a smaller subset for a faster lab execution
x_train = x_train[:10000]
y_train = y_train[:10000]
x_test = x_test[:5000]
y_test = y_test[:5000]

print("Training samples:", len(x_train))
print("Testing samples:", len(x_test))

## 3. Pad the Sequences

Movie reviews have different lengths. Neural networks normally need inputs of the same size, so we use **padding**.

Every review is converted to a sequence of exactly 200 integers.

In [ ]:
# Step 3: Pad sequences to the same length
x_train = keras.utils.pad_sequences(
    x_train, maxlen=MAX_LEN, padding="post", truncating="post"
)

x_test = keras.utils.pad_sequences(
    x_test, maxlen=MAX_LEN, padding="post", truncating="post"
)

print("Training shape:", x_train.shape)
print("Testing shape:", x_test.shape)

## 4. Build the RNN Model

The model contains three important stages:

1. **Embedding layer** – learns a numerical representation for words.
2. **SimpleRNN layer** – learns sequence patterns.
3. **Dense layer with sigmoid** – predicts positive or negative sentiment.

In [ ]:
# Step 4: Create the RNN model
model = keras.Sequential([
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=64, input_length=MAX_LEN),
    layers.SimpleRNN(64),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Train the RNN

`binary_crossentropy` is used because the problem has two classes: positive and negative.

The Adam optimizer updates the model weights during training.

In [ ]:
# Step 5: Train the RNN model
history = model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=3,
    batch_size=64
)

In [ ]:
# Step 6: Evaluate the model on test data
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

print("Test Loss:", round(float(test_loss), 4))
print("Test Accuracy:", round(float(test_accuracy * 100), 2), "%")

In [ ]:
# Step 7: Plot training and validation accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("RNN Sentiment Analysis Accuracy")
plt.legend()
plt.show()

## 6. Test the Model with New Reviews

Because the model expects the same integer representation used during training, we use the IMDB word index to convert new review words into integer sequences.

In [ ]:
# Step 8: Create a word-to-index dictionary for IMDB
word_index = keras.datasets.imdb.get_word_index()

# IMDB reserves the first few indexes for special tokens.
def encode_review(text):
    words = text.lower().replace(".", "").replace(",", "").split()
    encoded = [1]  # start token
    for word in words:
        index = word_index.get(word, 2) + 3
        if index >= VOCAB_SIZE:
            index = 2  # unknown word
        encoded.append(index)
    return keras.utils.pad_sequences(
        [encoded], maxlen=MAX_LEN, padding="post", truncating="post"
    )

In [ ]:
# Step 9: Predict sentiment for new reviews
reviews = [
    "This movie was excellent and I really enjoyed it",
    "The movie was boring and disappointing",
    "Amazing story and wonderful acting",
    "I hated this movie and it was terrible"
]

for review in reviews:
    sequence = encode_review(review)
    probability = float(model.predict(sequence, verbose=0)[0][0])
    sentiment = "Positive" if probability >= 0.5 else "Negative"
    print(f"Review: {review}")
    print(f"Sentiment: {sentiment} | Probability: {probability:.3f}")
    print("-" * 70)

## 7. Understanding the Result

If the sigmoid output is close to **1**, the model considers the review more likely to be positive.

If the output is close to **0**, the model considers the review more likely to be negative.

Example:

```text
Probability = 0.91 → Positive
Probability = 0.12 → Negative
```

The exact prediction may vary because neural-network training involves learned parameters and the training configuration.

## 8. Student Practice

Perform the following experiments:

1. Change the number of RNN units from `64` to `32` and compare accuracy.
2. Change the Embedding dimension from `64` to `128`.
3. Train for 5 epochs and compare the result.
4. Try adding a `Dropout` layer after the RNN.
5. Enter at least five of your own positive and negative sentences.

### Observation Table

| Model Configuration | Test Accuracy |
|---|---|
| SimpleRNN – 32 units | ______ |
| SimpleRNN – 64 units | ______ |
| SimpleRNN – 128 units | ______ |

## Result

Thus, **Sentiment Analysis using an RNN** was successfully implemented using TensorFlow/Keras. The model learned from movie reviews and classified new text as positive or negative sentiment.

## Viva Questions

1. What is Sentiment Analysis?
2. What is an RNN?
3. Why are RNNs useful for text data?
4. What is tokenization?
5. Why do we use padding?
6. What is an Embedding layer?
7. What is the purpose of the sigmoid activation function?
8. Why is binary cross-entropy used?
9. What is the IMDB dataset?
10. What is the difference between an RNN and a normal feed-forward network?